# LongCat-Video-Avatar 1.5 — A100 40GB/80GB '상처받음, 화남 아님' 여성 상담 영상 (v3.3)

이 노트북은 완전히 새로운 Google Colab 런타임을 기준으로 합니다.

- 입력: 여성 PNG 1개 + 여성 WAV 1개
- 생성: LongCat-Video-Avatar 1.5, INT8, 720p, 25fps
- A100 40GB: 공식 저장소 Low-VRAM PR의 순차 모델 로딩과 720p→480p OOM 자동 대체 적용
- A100 80GB: 일반 단일 GPU 모드 자동 적용
- 저장: 모델과 중간 파일은 `/content`, 입력 사본과 최종 MP4만 Google Drive
- 실행: **런타임 → 모두 실행**

실행 중 두 번만 사용자 입력이 필요합니다. 첫 번째는 PNG/WAV 업로드, 두 번째는 미리보기 시드 선택입니다.

업로드할 로컬 파일:

1. `family_center_platform_v0_4/frontend/public/personas/lee-jieun/neutral.png`
2. `family_center_platform_v0_4/backend/data/demo_media/demo-first-couple-conflict-01-lee-jieun-v3.wav`

In [ ]:
# 1. 드라이브 마운트, A100/공간 확인, 입력 파일 업로드
import os, re, math, json, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, files

drive.mount('/content/drive')

gpu_line = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'
]).decode().strip()
print('GPU:', gpu_line)
gpu_memory_mib = int(gpu_line.rsplit(',', 1)[1].strip())
if gpu_memory_mib < 38000:
    raise RuntimeError('최소 A100 40GB급 GPU가 필요합니다. 런타임 GPU를 다시 선택하세요.')
LOW_VRAM = gpu_memory_mib < 70000
print('실행 모드:', 'A100 40GB CPU 오프로딩' if LOW_VRAM else 'A100 80GB 일반 모드')

with open('/proc/meminfo', encoding='utf-8') as meminfo:
    mem_total_kib = int(re.search(r'MemTotal:\s+(\d+)', meminfo.read()).group(1))
ram_gib = mem_total_kib / 1024**2
print(f'시스템 RAM: {ram_gib:.1f} GiB')
if LOW_VRAM and ram_gib < 45:
    raise RuntimeError('A100 40GB 모드는 CPU 오프로딩을 위해 시스템 RAM 45GiB 이상이 필요합니다.')

usage = shutil.disk_usage('/content')
free_gib = usage.free / 1024**3
print(f'/content 빈 공간: {free_gib:.1f} GiB')
if free_gib < 120:
    raise RuntimeError('/content에 최소 120GiB의 빈 공간이 필요합니다.')

ROOT = Path('/content/longcat_avatar15')
INPUT_DIR = ROOT / 'inputs'
OUTPUT_DIR = ROOT / 'outputs'
DRIVE_DIR = Path('/content/drive/MyDrive/longcat_avatar15')
for folder in (INPUT_DIR, OUTPUT_DIR, DRIVE_DIR / 'inputs', DRIVE_DIR / 'outputs'):
    folder.mkdir(parents=True, exist_ok=True)

SOURCE_IMAGE = INPUT_DIR / 'lee_jieun_source_neutral.png'
SOURCE_AUDIO = INPUT_DIR / 'lee_jieun_speech_original.wav'
PADDED_AUDIO = INPUT_DIR / 'lee_jieun_speech_padded.wav'
DRIVE_IMAGE = DRIVE_DIR / 'inputs' / SOURCE_IMAGE.name
DRIVE_AUDIO = DRIVE_DIR / 'inputs' / SOURCE_AUDIO.name

need_image = not DRIVE_IMAGE.exists()
need_audio = not DRIVE_AUDIO.exists()
if need_image or need_audio:
    os.chdir('/content')
    requested = []
    if need_image:
        requested.append('lee-jieun/neutral.png')
    if need_audio:
        requested.append('demo-first-couple-conflict-01-lee-jieun-v3.wav')
    print('\n다음 파일만 선택하세요:', ', '.join(requested))
    uploaded = files.upload()
    uploaded_paths = [Path('/content') / name for name in uploaded.keys()]
    images = [p for p in uploaded_paths if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}]
    audios = [p for p in uploaded_paths if p.suffix.lower() in {'.wav', '.mp3', '.m4a'}]
    if need_image:
        assert len(images) == 1, f'neutral.png 이미지를 정확히 1개 올려주세요: {images}'
        shutil.copy2(images[0], DRIVE_IMAGE)
    if need_audio:
        assert len(audios) == 1, f'음성을 정확히 1개 올려주세요: {audios}'
        shutil.copy2(audios[0], DRIVE_AUDIO)
else:
    print('기존 Drive 중립 이미지와 음성을 재사용합니다.')

shutil.copy2(DRIVE_IMAGE, SOURCE_IMAGE)
shutil.copy2(DRIVE_AUDIO, SOURCE_AUDIO)

subprocess.run([
    'ffmpeg', '-y', '-v', 'warning', '-i', str(SOURCE_AUDIO),
    '-af', 'adelay=600:all=1,apad=pad_dur=0.6',
    '-ar', '16000', '-ac', '1', '-c:a', 'pcm_s16le', str(PADDED_AUDIO)
], check=True)
print('입력 준비 완료:')
print(' ', SOURCE_IMAGE)
print(' ', PADDED_AUDIO)

In [ ]:
# 2. 공식 코드와 전용 Python 3.10 환경 설치, 필요한 가중치만 다운로드
def run(command, *, cwd=None, env=None):
    print('\n$', ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

run(['apt-get', '-qq', 'update'])
run(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'git', 'git-lfs', 'libsndfile1'])
run(['git', 'lfs', 'install'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

REPO = ROOT / 'repo'
ENV_DIR = ROOT / '.venv'
PYTHON = ENV_DIR / 'bin' / 'python'
HF = ENV_DIR / 'bin' / 'hf'
WEIGHTS = ROOT / 'weights'
BASE_MODEL = WEIGHTS / 'LongCat-Video'
AVATAR_MODEL = WEIGHTS / 'LongCat-Video-Avatar-1.5'

if not REPO.exists():
    run(['git', 'clone', '--single-branch', '--branch', 'main',
         'https://github.com/meituan-longcat/LongCat-Video', str(REPO)])

if not PYTHON.exists():
    run(['uv', 'venv', '--python', '3.10', str(ENV_DIR)])

run(['uv', 'pip', 'install', '--python', str(PYTHON),
     'torch==2.6.0+cu124', 'torchvision==0.21.0+cu124', 'torchaudio==2.6.0+cu124',
     '--index-url', 'https://download.pytorch.org/whl/cu124'])

requirements = (REPO / 'requirements.txt').read_text(encoding='utf-8').splitlines()
requirements = [line for line in requirements if line.strip()
                and not line.startswith('torch==')
                and not line.startswith('flash-attn==')]
filtered_requirements = ROOT / 'requirements_colab.txt'
filtered_requirements.write_text('\n'.join(requirements) + '\n', encoding='utf-8')

run(['uv', 'pip', 'install', '--python', str(PYTHON), '-r', str(filtered_requirements)])
avatar_requirements = (REPO / 'requirements_avatar.txt').read_text(encoding='utf-8').splitlines()
avatar_requirements = [line for line in avatar_requirements if line.strip()
                       and not line.startswith('libsndfile1==')
                       and not line.startswith('tritonserverclient==')
                       and not line.startswith('openai==')]
filtered_avatar_requirements = ROOT / 'requirements_avatar_colab.txt'
filtered_avatar_requirements.write_text('\n'.join(avatar_requirements) + '\n', encoding='utf-8')
run(['uv', 'pip', 'install', '--python', str(PYTHON), '-r', str(filtered_avatar_requirements)])
run(['uv', 'pip', 'install', '--python', str(PYTHON),
     'huggingface_hub[cli]', 'ninja', 'packaging', 'wheel', 'setuptools'])

flash_env = os.environ.copy()
flash_env['MAX_JOBS'] = '4'
flash_env['TORCH_CUDA_ARCH_LIST'] = '8.0'
run(['uv', 'pip', 'install', '--python', str(PYTHON),
     'flash-attn==2.7.4.post1', '--no-build-isolation'], env=flash_env)

model_env = os.environ.copy()
model_env['HF_HOME'] = '/content/hf_cache'
BASE_MODEL.mkdir(parents=True, exist_ok=True)
AVATAR_MODEL.mkdir(parents=True, exist_ok=True)

run([str(HF), 'download', 'meituan-longcat/LongCat-Video',
     '--include', 'tokenizer/*', 'text_encoder/*', 'vae/*', 'config.json', 'model_index.json',
     '--local-dir', str(BASE_MODEL)], env=model_env)
run([str(HF), 'download', 'meituan-longcat/LongCat-Video-Avatar-1.5',
     '--include', 'base_model_int8/*', 'lora/*', 'scheduler/*',
     'vocal_separator/*', 'whisper-large-v3/*', 'config.json', 'model_index.json',
     '--local-dir', str(AVATAR_MODEL)], env=model_env)

run([str(PYTHON), '-c',
     "import torch, flash_attn; print('torch', torch.__version__); print(torch.cuda.get_device_name(0))"])

# 공식 저장소 PR #115의 저메모리 단일 GPU 실행 파일만 가져온다.
LOWMEM_SCRIPT = REPO / 'run_demo_avatar_single_lowmem.py'
run(['git', 'fetch', 'origin', '+pull/115/head:lowmem-pr'], cwd=str(REPO))
lowmem_source = subprocess.check_output(
    ['git', 'show', 'lowmem-pr:run_demo_avatar_single_lowmem.py'], cwd=str(REPO)
)
LOWMEM_SCRIPT.write_bytes(lowmem_source)
print('\n설치와 모델 다운로드 완료')

In [ ]:
# 3. 자연스러운 상담 장면 지시문과 실행 작업 생성
PROMPT = '''
Locked-off medium close-up in a quiet family counseling room.
A Korean woman in her mid-thirties speaks softly to the counselor. She feels
emotionally hurt and quietly disappointed, never angry, irritated, confrontational,
accusing, or resentful. The performance is intimate, vulnerable, and underplayed.

Hurt is communicated only through a slightly softened gaze, a barely perceptible
upward lift of the inner eyebrows, gentle eyelid heaviness, and subtle lip tension.
The eyebrows never pull downward or strongly together. The glabella stays smooth.
The eyes never glare or widen. The nostrils never flare. The jaw never clenches.
Her expression remains close to the neutral source portrait and does not intensify
or accumulate across the shot.

Speech produces accurate lip shapes and realistic jaw and chin movement, with only
very small supporting motion in the lower cheeks. Upper-cheek, nose, forehead, and
eyebrow movement remain minimal. Her mouth never stretches into a grimace.

She makes two or three slow, irregular natural blinks, tiny eye refocusing movements,
quiet breathing, and at most one barely perceptible coherent head adjustment. The
crown, hairline, ears, neck, and shoulders remain physically connected and stable.
Her mouth gradually returns to a fully closed resting position during every silence.

No anger cues, downward knitted eyebrows, scowl, hard stare, facial tension, emotional
escalation, repeated expression cycle, repeated nodding, body swaying, hand gestures,
camera movement, zoom, face warping, rubbery skin, identity drift, sudden pose change,
crying performance, melodrama, or exaggerated acting.
'''.strip()

JOB_JSON = ROOT / 'lee_jieun_job.json'
JOB_JSON.write_text(json.dumps({
    'prompt': PROMPT,
    'cond_image': str(SOURCE_IMAGE),
    'cond_audio': {'person1': str(PADDED_AUDIO)}
}, ensure_ascii=False, indent=2), encoding='utf-8')

# 공식 스크립트의 고정 시드를 환경변수로 받을 수 있게 변경
script_path = REPO / 'run_demo_avatar_single_audio_to_video.py'
script_text = script_path.read_text(encoding='utf-8')
old_seed = 'global_seed = 42'
new_seed = "global_seed = int(os.environ.get('LONGCAT_SEED', '42'))"
if old_seed in script_text:
    script_path.write_text(script_text.replace(old_seed, new_seed), encoding='utf-8')
elif new_seed not in script_text:
    raise RuntimeError('공식 스크립트 구조가 변경되어 시드 설정에 실패했습니다.')

# Low-VRAM PR 스크립트도 미리보기마다 다른 시드를 사용하게 한다.
lowmem_text = LOWMEM_SCRIPT.read_text(encoding='utf-8')
lowmem_old_seed = 'generator.manual_seed(42 + global_rank)'
lowmem_new_seed = "generator.manual_seed(int(os.environ.get('LONGCAT_SEED', '42')) + global_rank)"
if lowmem_old_seed in lowmem_text:
    lowmem_text = lowmem_text.replace(lowmem_old_seed, lowmem_new_seed)
elif lowmem_new_seed not in lowmem_text:
    raise RuntimeError('Low-VRAM 스크립트 시드 설정에 실패했습니다.')

# 720p 연속 구간에서 KV 캐시를 CPU로 내려 A100 40GB의 피크 VRAM을 줄인다.
lowmem_text = lowmem_text.replace('offload_kv_cache=False', 'offload_kv_cache=True')

# 구간 MP4에 임시 오디오가 들어 있어도 최종 파일에는 전체 원본 WAV만 선택한다.
# 이 명시적 map이 없으면 각 3.2초 구간마다 음성이 처음부터 반복된다.
lowmem_text = lowmem_text.replace(
    '-c:v copy -c:a aac -shortest',
    '-map 0:v:0 -map 1:a:0 -c:v copy -c:a aac -shortest'
)
LOWMEM_SCRIPT.write_text(lowmem_text, encoding='utf-8')

print(JOB_JSON.read_text(encoding='utf-8'))

In [ ]:
# 4. 3.7초 미리보기 3개 생성 및 재생
from IPython.display import Video, display

PREVIEW_SEEDS = [17321, 29411, 61703]
RESOLUTION = '720p'

def avatar_command(output_dir, seed, num_segments, *, ref_img_index=None, mask_frame_range=None):
    if LOW_VRAM:
        command = [
            str(PYTHON), '-m', 'torch.distributed.run', '--nproc_per_node=1',
            str(LOWMEM_SCRIPT),
            '--checkpoint_dir', str(AVATAR_MODEL),
            '--stage_1', 'ai2v',
            '--input_json', str(JOB_JSON),
            '--resolution', RESOLUTION,
            '--num_segments', str(num_segments),
            '--output_dir', str(output_dir),
        ]
    else:
        command = [
            str(PYTHON), '-m', 'torch.distributed.run', '--nproc_per_node=1',
            str(REPO / 'run_demo_avatar_single_audio_to_video.py'),
            '--context_parallel_size=1',
            '--checkpoint_dir', str(AVATAR_MODEL),
            '--stage_1', 'ai2v',
            '--input_json', str(JOB_JSON),
            '--model_type', 'avatar-v1.5',
            '--use_distill', '--use_int8',
            '--resolution', RESOLUTION,
            '--num_segments', str(num_segments),
            '--output_dir', str(output_dir),
        ]
    if ref_img_index is not None:
        command += ['--ref_img_index', str(ref_img_index)]
    if mask_frame_range is not None:
        command += ['--mask_frame_range', str(mask_frame_range)]
    environment = os.environ.copy()
    environment['LONGCAT_SEED'] = str(seed)
    environment['PYTHONPATH'] = str(REPO)
    return command, environment

for seed in PREVIEW_SEEDS:
    preview_dir = OUTPUT_DIR / f'preview_hurt_not_angry_seed_{seed}'
    preview_dir.mkdir(parents=True, exist_ok=True)
    preview_file = preview_dir / ('segment_001.mp4' if LOW_VRAM else 'ai2v_demo_1.mp4')
    if not preview_file.exists():
        command, environment = avatar_command(preview_dir, seed, 1)
        print(f'\n===== 미리보기 시드 {seed} 생성 =====')
        try:
            subprocess.run(command, cwd=str(REPO), env=environment, check=True)
        except subprocess.CalledProcessError:
            if LOW_VRAM and RESOLUTION == '720p':
                print('720p가 GPU 메모리를 초과하여 480p로 자동 재시도합니다.')
                RESOLUTION = '480p'
                shutil.rmtree(preview_dir, ignore_errors=True)
                preview_dir.mkdir(parents=True, exist_ok=True)
                command, environment = avatar_command(preview_dir, seed, 1)
                subprocess.run(command, cwd=str(REPO), env=environment, check=True)
            else:
                raise
    print(f'\n시드 {seed}')
    display(Video(str(preview_file), embed=True, width=900))

print('세 영상 중 가장 자연스러운 시드 번호를 다음 셀에 입력하세요.')

In [ ]:
# 5. 선택한 시드로 전체 영상 렌더링, Drive 저장, 재생
import soundfile as sf

BEST_SEED = int(input(f'가장 자연스러운 시드를 입력하세요 {PREVIEW_SEEDS}: ').strip())
if BEST_SEED not in PREVIEW_SEEDS:
    raise ValueError(f'{PREVIEW_SEEDS} 중 하나를 입력해야 합니다.')

audio_data, sample_rate = sf.read(str(PADDED_AUDIO))
audio_duration = len(audio_data) / sample_rate
first_segment_seconds = 93 / 25
following_segment_seconds = (93 - 13) / 25
NUM_SEGMENTS = max(1, 1 + math.ceil(
    max(0, audio_duration - first_segment_seconds) / following_segment_seconds
))
print(f'음성 길이: {audio_duration:.2f}초 / 생성 구간: {NUM_SEGMENTS}')

# A100 40GB에서는 720p 연속 구간이 미리보기보다 VRAM을 더 사용한다.
# 먼저 KV 캐시 CPU 오프로딩으로 720p를 시도하고, 실패할 때만 480p로 자동 재시도한다.
def run_full_render(render_resolution):
    global RESOLUTION
    RESOLUTION = render_resolution
    render_output = OUTPUT_DIR / f'full_hurt_not_angry_seed_{BEST_SEED}_{render_resolution}'
    render_output.mkdir(parents=True, exist_ok=True)
    expected = render_output / ('final_video.mp4' if LOW_VRAM and NUM_SEGMENTS > 1 else 'segment_001.mp4')
    if expected.exists() and expected.stat().st_size > 100_000:
        print(f'기존 완성본 재사용: {expected}')
        return render_output

    command, environment = avatar_command(
        render_output, BEST_SEED, NUM_SEGMENTS, ref_img_index=0, mask_frame_range=3
    )
    environment['PYTHONUNBUFFERED'] = '1'
    log_path = render_output / 'render.log'
    print(f'\n===== 전체 영상 {render_resolution} 생성 =====')
    print('실패 시 상세 로그:', log_path)
    with log_path.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            command, cwd=str(REPO), env=environment,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1
        )
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    if return_code != 0:
        print(f'{render_resolution} 실패 (종료 코드 {return_code})')
        return None
    return render_output

full_output = run_full_render('720p')
if full_output is None and LOW_VRAM:
    print('\n720p 연속 렌더가 40GB를 초과했습니다. 480p로 자동 재시도합니다.')
    full_output = run_full_render('480p')
if full_output is None:
    logs = sorted(OUTPUT_DIR.glob(f'full_hurt_not_angry_seed_{BEST_SEED}_*/render.log'), key=lambda p: p.stat().st_mtime)
    if logs:
        last_log = logs[-1]
        drive_log = DRIVE_DIR / 'outputs' / 'last_render_error.log'
        shutil.copy2(last_log, drive_log)
        tail = last_log.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]
        print('\n===== 실제 오류 마지막 80줄 =====')
        print('\n'.join(tail))
        print('오류 로그 Drive 저장:', drive_log)
    raise RuntimeError('720p와 480p 전체 렌더가 모두 실패했습니다. 위 실제 오류를 보내주세요.')

if LOW_VRAM:
    generated_final = full_output / ('final_video.mp4' if NUM_SEGMENTS > 1 else 'segment_001.mp4')
else:
    continued = list(full_output.glob('video_continue_*.mp4'))
    if continued:
        def segment_number(path):
            match = re.search(r'video_continue_(\d+)', path.stem)
            return int(match.group(1)) if match else 0
        generated_final = max(continued, key=segment_number)
    else:
        generated_final = full_output / 'ai2v_demo_1.mp4'

if not generated_final.exists():
    raise FileNotFoundError(f'최종 영상이 생성되지 않았습니다: {generated_final}')

DRIVE_FINAL = DRIVE_DIR / 'outputs' / 'lee_jieun_longcat15_hurt_not_angry.mp4'
shutil.copy2(generated_final, DRIVE_FINAL)
print(f'\n완료: {DRIVE_FINAL}')
print(f'파일 크기: {DRIVE_FINAL.stat().st_size / 1024**2:.1f} MB')
display(Video(str(DRIVE_FINAL), embed=True, width=1000))

In [ ]:
# 6. 선택 사항: 최종 MP4를 현재 PC로 다운로드
from google.colab import files
files.download(str(DRIVE_FINAL))